# Setup

In [ ]:
# ============================================================
# SETUP CELL — run once per session
# All shared functions, no experiment logic
# ============================================================

import random, os, json, time
import numpy as np
import torch
from datetime import datetime
from google.colab import drive
import json, os
import matplotlib.pyplot as plt

drive.mount('/content/drive')
SAVE_DIR  = '/content/drive/MyDrive/MKP_results'
CACHE_DIR = '/content/drive/MyDrive/MKP_results/pj_cache'
os.makedirs(SAVE_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — IS estimator will be very slow.")
    print("Go to Runtime → Change runtime type → T4 GPU before continuing.")

# ── Serialization ─────────────────────────────────────────────
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_):    return bool(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

def save_json(data, path):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2, cls=NumpyEncoder)
    print(f"  Saved: {path}")

# ── Instance ──────────────────────────────────────────────────
def generate_instance(n, m, alpha, seed=42):
    np.random.seed(seed)
    values     = np.random.randint(1, 100, n)
    weights    = np.random.randint(1,  50, (m, n))
    capacities = np.array([int(alpha * weights[i].sum()) for i in range(m)])
    return values, weights, capacities

# ── p_j cache ─────────────────────────────────────────────────
def get_pj(weights, capacities, n_samples,
           n, m, alpha, instance_seed, pj_seed=42,
           force_recompute=False):
    alpha_str = str(alpha).replace('.', '')
    fname     = f"pj_n{n}_m{m}_a{alpha_str}_seed{instance_seed}_{n_samples//1_000_000}M.json"
    path      = os.path.join(CACHE_DIR, fname)
    if not force_recompute and os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        pj = np.array(d['pj'], dtype=np.float32)
        print(f"  p_j loaded: {fname}")
        print(f"  [{pj.min():.3f},{pj.max():.3f}] mean={pj.mean():.3f}")
        return pj
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Computing p_j ({n_samples//1_000_000}M samples) on {device}...")
    W = torch.tensor(weights,    dtype=torch.float32, device=device)
    C = torch.tensor(capacities, dtype=torch.float32, device=device)
    torch.manual_seed(pj_seed)
    count_in  = torch.zeros(n, device=device)
    count_out = torch.zeros(n, device=device)
    total_in  = torch.zeros(n, device=device)
    total_out = torch.zeros(n, device=device)
    processed = 0
    t0 = time.time()
    while processed < n_samples:
        bs    = min(50_000, n_samples - processed)
        X     = torch.randint(0, 2, (bs, n), device=device, dtype=torch.float32)
        usage = X @ W.T
        for j in range(n):
            in_j  = X[:, j] == 1
            out_j = X[:, j] == 0
            uwj   = usage - X[:, j:j+1] * W[:, j]
            feas_in  = (uwj <= C - W[:, j]).all(dim=1) & in_j
            feas_out = (uwj <= C          ).all(dim=1) & out_j
            count_in[j]  += feas_in.sum()
            count_out[j] += feas_out.sum()
            total_in[j]  += in_j.sum()
            total_out[j] += out_j.sum()
        processed += bs
    ri = (count_in  / total_in.clamp(min=1) ).cpu().numpy()
    ro = (count_out / total_out.clamp(min=1)).cpu().numpy()
    pj = ri / (ri + ro + 1e-12)
    print(f"  Done in {time.time()-t0:.1f}s | "
          f"[{pj.min():.3f},{pj.max():.3f}] mean={pj.mean():.3f}")
    with open(path, 'w') as f:
        json.dump({'pj': pj.tolist(), 'n': n, 'm': m,
                   'alpha': float(alpha), 'instance_seed': instance_seed,
                   'pj_seed': pj_seed, 'n_samples': n_samples}, f)
    print(f"  Cached: {fname}")
    return pj

# ── Feasibility + repair ───────────────────────────────────────
def is_feasible(sol, weights, capacities, n, m):
    return all(
        sum(weights[i][j] * sol[j] for j in range(n)) <= capacities[i]
        for i in range(m)
    )

def toyoda_heuristic(values, weights, capacities, n, m):
    scores = []
    for j in range(n):
        denom = sum(weights[i][j] / capacities[i] for i in range(m))
        scores.append((values[j] / denom if denom > 0 else 0, j))
    scores.sort(reverse=True)
    sol = [0] * n
    rem = list(capacities)
    for _, j in scores:
        if all(rem[i] >= weights[i][j] for i in range(m)):
            sol[j] = 1
            for i in range(m): rem[i] -= weights[i][j]
    return float(sum(sol) / n)

def repair_solution(sol, values, weights, capacities, n, m):
    scores = []
    for j in range(n):
        denom = sum(weights[i][j] / capacities[i] for i in range(m))
        scores.append((values[j] / denom if denom > 0 else 0, j))
    scores.sort(reverse=True)
    order = [j for _, j in scores]
    sol   = list(sol)
    for j in order:
        if is_feasible(sol, weights, capacities, n, m): break
        sol[j] = 0
    for j in reversed(order):
        if sol[j] == 0:
            sol[j] = 1
            if not is_feasible(sol, weights, capacities, n, m):
                sol[j] = 0
    return sol

def compute_ref_best(n, m, vl, wl, cl,
                     pop_size=50, max_offs=20000, n_runs=10):
    print(f"  Computing reference best...")
    best_overall = 0
    for run in range(n_runs):
        random.seed(run)
        pop  = [[random.randint(0,1) for _ in range(n)]
                for _ in range(pop_size)]
        pop  = [repair_solution(s, vl, wl, cl, n, m) for s in pop]
        fits = [sum(vl[j]*s[j] for j in range(n)) for s in pop]
        best = max(fits)
        for _ in range(max_offs):
            a,b = random.randint(0,pop_size-1), random.randint(0,pop_size-1)
            p1  = pop[a] if fits[a]>=fits[b] else pop[b]
            a,b = random.randint(0,pop_size-1), random.randint(0,pop_size-1)
            p2  = pop[a] if fits[a]>=fits[b] else pop[b]
            c_  = [p1[j] if random.random()<0.5 else p2[j] for j in range(n)]
            c_  = [1-c_[j] if random.random()<1/n else c_[j] for j in range(n)]
            c_  = repair_solution(c_, vl, wl, cl, n, m)
            cf  = sum(vl[j]*c_[j] for j in range(n))
            wi  = fits.index(min(fits))
            if cf > fits[wi]:
                pop[wi]  = c_
                fits[wi] = cf
            if cf > best: best = cf
        best_overall = max(best_overall, best)
    print(f"  Reference best = {best_overall}")
    return int(best_overall)

# ── Population builders ────────────────────────────────────────
def make_pop(method, pop_size, n, rng,
             pj=None, hill_p=None,
             values=None, weights=None, capacities=None,
             m=None, n_candidates=400):
    if method == 'uniform':
        return [[rng.randint(0,1) for _ in range(n)]
                for _ in range(pop_size)]
    elif method == 'hill':
        p = hill_p or 0.5
        return [[1 if rng.random() < p else 0 for _ in range(n)]
                for _ in range(pop_size)]
    elif method == 'gf':
        return [[1 if rng.random() < pj[j] else 0 for j in range(n)]
                for _ in range(pop_size)]
    elif method == 'gf_filtered':
        feasible_pool = []
        total = 0
        while len(feasible_pool) < pop_size:
            batch = [[1 if rng.random() < pj[j] else 0 for j in range(n)]
                     for _ in range(n_candidates)]
            total += n_candidates
            for sol in batch:
                if is_feasible(sol, weights, capacities, n, m):
                    feasible_pool.append(
                        (sum(values[j]*sol[j] for j in range(n)), sol))
            if total > n_candidates * 20 and len(feasible_pool) == 0:
                print("    WARNING: quality filter fallback to standard gf")
                return make_pop('gf', pop_size, n, rng, pj=pj)
        feasible_pool.sort(key=lambda x: x[0], reverse=True)
        return [s for _, s in feasible_pool[:pop_size]]
    elif method == 'gf_diverse':
        # diversity-aware filter: top quality + min hamming distance
        feasible_pool = []
        total = 0
        min_hamming = n_candidates  # reuse n_candidates param as min distance
        while len(feasible_pool) < pop_size * 4:
            batch = [[1 if rng.random() < pj[j] else 0 for j in range(n)]
                     for _ in range(400)]
            total += 400
            for sol in batch:
                if is_feasible(sol, weights, capacities, n, m):
                    feasible_pool.append(
                        (sum(values[j]*sol[j] for j in range(n)), sol))
            if total > 8000 and len(feasible_pool) == 0:
                return make_pop('gf', pop_size, n, rng, pj=pj)
        feasible_pool.sort(key=lambda x: x[0], reverse=True)
        selected = [feasible_pool[0][1]]
        for _, sol in feasible_pool[1:]:
            if len(selected) >= pop_size: break
            min_dist = min(
                sum(a != b for a, b in zip(sol, s))
                for s in selected
            )
            if min_dist >= min_hamming:
                selected.append(sol)
        while len(selected) < pop_size:
            selected.append(feasible_pool[len(selected)][1])
        return selected
    elif method == 'gf_filtered_hybrid':
        nb = pop_size // 2
        filtered = make_pop('gf_filtered', nb, n, rng,
                            pj=pj, values=values,
                            weights=weights, capacities=capacities,
                            m=m, n_candidates=n_candidates)
        uniform  = make_pop('uniform', pop_size - nb, n, rng)
        return filtered + uniform

# ── GA runner ──────────────────────────────────────────────────
def run_one_ga(n, m, values, weights, capacities,
               method, penalty_type, max_offspring,
               pj=None, hill_p=None, ref=None,
               n_candidates=400, seed=0,
               track_diversity=False, track_every=50):
    random.seed(seed)
    rng = random.Random(seed)
    pop = make_pop(method, 50, n, rng,
                   pj=pj, hill_p=hill_p,
                   values=values, weights=weights,
                   capacities=capacities, m=m,
                   n_candidates=n_candidates)

    PENALTY_MULT = 50.0
    CONV_THRESH  = [50, 60, 70, 80, 90, 95, 98, 99]
    POP_SIZE     = len(pop)

    def fit(sol):
        obj = sum(values[j]*sol[j] for j in range(n))
        if penalty_type == 'death':
            return obj if is_feasible(sol, weights, capacities, n, m) else 0
        viol = sum(
            max(0, sum(weights[i][j]*sol[j] for j in range(n))
                - capacities[i])
            for i in range(m)
        )
        return obj - PENALTY_MULT * viol

    def hamming_div(p):
        total = count = 0
        for i in range(len(p)):
            for j in range(i+1, len(p)):
                total += sum(a != b for a, b in zip(p[i], p[j]))
                count += 1
        return float(total/count) if count else 0.0

    fits  = [fit(s) for s in pop]
    feas0 = [sum(values[j]*s[j] for j in range(n))
             for s in pop
             if is_feasible(s, weights, capacities, n, m)]
    gen0_feas = float(len(feas0) / POP_SIZE)
    gen0_best = float(max(feas0)) if feas0 else 0.0
    gen0_div  = hamming_div(pop) if track_diversity else 0.0

    best     = max(fits)
    best_sol = pop[fits.index(best)].copy()
    conv     = {str(t): None for t in CONV_THRESH}
    curve    = [[0, float(best), float(np.mean(fits)), gen0_div]]

    for oc in range(1, max_offspring + 1):
        a,b = random.randint(0,POP_SIZE-1), random.randint(0,POP_SIZE-1)
        p1  = pop[a] if fits[a] >= fits[b] else pop[b]
        a,b = random.randint(0,POP_SIZE-1), random.randint(0,POP_SIZE-1)
        p2  = pop[a] if fits[a] >= fits[b] else pop[b]
        c_  = [p1[j] if random.random()<0.5 else p2[j] for j in range(n)]
        c_  = [1-c_[j] if random.random()<1/n else c_[j] for j in range(n)]
        cf  = fit(c_)
        wi  = fits.index(min(fits))
        if cf > fits[wi]:
            pop[wi]  = c_
            fits[wi] = cf
        if cf > best:
            best     = cf
            best_sol = c_.copy()
        if ref:
            for t in CONV_THRESH:
                if conv[str(t)] is None and best >= (t/100)*ref:
                    conv[str(t)] = int(oc)
        if oc % track_every == 0:
            div = hamming_div(pop) if track_diversity else 0.0
            curve.append([int(oc), float(best),
                          float(np.mean(fits)), div])

    best_obj  = int(sum(values[j]*best_sol[j] for j in range(n)))
    best_feas = bool(is_feasible(best_sol, weights, capacities, n, m))

    return {
        'feasibility_rate_gen0':  gen0_feas,
        'pct_of_best_gen0':       float(gen0_best/ref*100) if ref else 0.0,
        'diversity_gen0':         gen0_div,
        'best_fitness_final':     best_obj,
        'best_solution_feasible': best_feas,
        'pct_of_best_final':      float(best_obj/ref*100) if ref else 0.0,
        'convergence':            conv,
        'curve':                  curve,
    }

def run_method(label, method, penalty, n_runs, max_offspring,
               n, m, vl, wl, cl, pj, hill_p, ref,
               n_candidates=400, track_diversity=False):
    runs = []
    for r in range(n_runs):
        res = run_one_ga(
            n, m, vl, wl, cl,
            method=method, penalty_type=penalty,
            max_offspring=max_offspring,
            pj=pj, hill_p=hill_p, ref=ref,
            n_candidates=n_candidates,
            track_diversity=track_diversity,
            seed=r)
        runs.append(res)
    finals = [r['pct_of_best_final'] for r in runs]
    gen0s  = [r['feasibility_rate_gen0']*100 for r in runs]
    c90s   = [r['convergence'].get('90') for r in runs
              if r['convergence'].get('90')]
    c90str = f"{np.mean(c90s):.0f}" if c90s else "never"
    print(f"  {label:>24}: "
          f"final={np.mean(finals):>6.1f}%  "
          f"gen0={np.mean(gen0s):>5.1f}%  "
          f"conv90={c90str}")
    return runs
def compute_pj_IS(weights, capacities,
                  n_samples=10_000_000, q=None, batch_size=200_000,
                  verbose=True):
    """
    Importance-sampling p_j estimator.

    weights:    (m, n) array
    capacities: (m,) array
    n_samples:  Monte Carlo budget
    q:          Bernoulli proposal probability. None -> mean alpha.
    batch_size: GPU batch size

    Returns dict with: pj, rho_inc, rho_exc, ess_mean, ess_min,
                       q_used, time_sec, feas_rate_proposal.
    """
    W = np.asarray(weights, dtype=np.float64)
    C = np.asarray(capacities, dtype=np.float64)
    m, n = W.shape

    alphas = C / W.sum(axis=1)
    if q is None:
        q = float(alphas.mean())
    q = float(np.clip(q, 1e-3, 1 - 1e-3))

    if verbose:
        print(f"\n  [compute_pj_IS] n={n}, m={m}, n_samples={n_samples:,}")
        print(f"    alphas: min={alphas.min():.3f}  "
              f"mean={alphas.mean():.3f}  max={alphas.max():.3f}")
        print(f"    q (proposal) = {q:.4f}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    W_t = torch.tensor(W, dtype=torch.float64, device=device)
    C_t = torch.tensor(C, dtype=torch.float64, device=device)

    log_q   = float(np.log(q))
    log_1mq = float(np.log(1.0 - q))

    sum_w_inc     = torch.zeros(n, dtype=torch.float64, device=device)
    sum_w_exc     = torch.zeros(n, dtype=torch.float64, device=device)
    sum_w_total   = torch.zeros((), dtype=torch.float64, device=device)
    feas_proposal = torch.zeros((), dtype=torch.float64, device=device)

    n_batches = (n_samples + batch_size - 1) // batch_size
    samples_done = 0
    ess_list = []
    t0 = time.time()

    for b in range(n_batches):
        bs = min(batch_size, n_samples - samples_done)

        # Bernoulli(q) sample on n bits
        s = (torch.rand(bs, n, device=device) < q).double()
        k = s.sum(dim=1)

        # Log IS weight, stabilized; constants drop in self-normalization
        log_w = -(k * log_q + (n - k) * log_1mq)
        log_w = log_w - log_w.max()
        w = torch.exp(log_w)

        # Constraint totals: T = s @ W^T
        T = s @ W_t.T  # (bs, m)

        # Memory-efficient AND across constraints — O(bs * n)
        cond_inc  = torch.ones(bs, n, dtype=torch.bool, device=device)
        cond_exc  = torch.ones(bs, n, dtype=torch.bool, device=device)
        feas_full = torch.ones(bs,    dtype=torch.bool, device=device)
        for i in range(m):
            T_i = T[:, i:i+1]
            W_i = W_t[i:i+1, :]
            C_i = float(C_t[i].item())
            cond_inc  &= (T_i <= C_i - W_i * (1 - s))
            cond_exc  &= (T_i <= C_i + W_i * s)
            feas_full &= (T[:, i] <= C_i)

        w_e = w.unsqueeze(1)
        sum_w_inc     += (w_e * cond_inc.double()).sum(dim=0)
        sum_w_exc     += (w_e * cond_exc.double()).sum(dim=0)
        sum_w_total   += w.sum()
        feas_proposal += feas_full.double().sum()

        ess = (w.sum() ** 2 / (w ** 2).sum()).item()
        ess_list.append(ess)
        samples_done += bs

        if verbose and (b == 0 or (b + 1) % max(1, n_batches // 5) == 0
                        or b == n_batches - 1):
            print(f"    batch {b+1:>3}/{n_batches}  "
                  f"ESS={ess:>11,.0f}/{bs:,}  ({100*ess/bs:5.2f}%)")

    sw_inc = sum_w_inc.cpu().numpy()
    sw_exc = sum_w_exc.cpu().numpy()
    sw_tot = float(sum_w_total.cpu().item())

    rho_inc = sw_inc / sw_tot
    rho_exc = sw_exc / sw_tot
    eps = 1e-12
    pj = (sw_inc + eps) / (sw_inc + sw_exc + 2 * eps)

    elapsed   = time.time() - t0
    feas_rate = float(feas_proposal.cpu().item()) / n_samples

    out = {
        'pj':                 pj.astype(np.float32),
        'rho_inc':            rho_inc,
        'rho_exc':            rho_exc,
        'ess_mean':           float(np.mean(ess_list)),
        'ess_min':            float(np.min(ess_list)),
        'q_used':             q,
        'time_sec':           elapsed,
        'n_samples':          n_samples,
        'feas_rate_proposal': feas_rate,
    }

    if verbose:
        print(f"    done in {elapsed:.1f}s")
        print(f"    ESS mean: {out['ess_mean']:>11,.0f}  "
              f"({100*out['ess_mean']/batch_size:5.2f}% of batch)")
        print(f"    ESS min : {out['ess_min']:>11,.0f}")
        print(f"    proposal feasibility: {100*feas_rate:5.2f}%")
        print(f"    p_j: min={pj.min():.3f}  mean={pj.mean():.3f}  "
              f"max={pj.max():.3f}  std={pj.std():.3f}")

    return out


def feasibility_rate_check(p_j, weights, capacities, n_pop=10_000, seed=0):
    """% of Bernoulli(p_j) samples that satisfy all constraints."""
    p_j = np.asarray(p_j, dtype=np.float64)
    W   = np.asarray(weights, dtype=np.float64)
    C   = np.asarray(capacities, dtype=np.float64)
    n   = len(p_j)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    P   = torch.tensor(p_j, dtype=torch.float64, device=device)
    W_t = torch.tensor(W, dtype=torch.float64, device=device)
    C_t = torch.tensor(C, dtype=torch.float64, device=device)

    g = torch.Generator(device=device).manual_seed(seed)
    s = (torch.rand(n_pop, n, device=device, generator=g) < P).double()
    T = s @ W_t.T
    return float((T <= C_t).all(dim=1).double().mean().item())

print("Setup complete. Functions ready:")
print("  generate_instance, get_pj, is_feasible, toyoda_heuristic,")
print("  repair_solution, compute_ref_best, make_pop, run_one_ga, run_method")
print()
print("Methods available in make_pop:")
print("  uniform, hill, gf, gf_filtered, gf_diverse, gf_filtered_hybrid")

# Experiments

## Budget Sweep

In [ ]:
# ── Experiment: Budget Sweep ──────────────────────────────────
N, M, ALPHA   = 100, 3, 0.25
REF_BEST      = 2375
INSTANCE_SEED = 42
PJ_SAMPLES    = 20_000_000
N_RUNS        = 20
BUDGET_VALUES = [200, 500, 1000, 2000, 5000, 10000, 20000]
timestamp     = datetime.now().strftime("%Y%m%d_%H%M%S")

METHODS = [
    ('uniform_soft',      'uniform',     'soft'),
    ('hill_soft',         'hill',        'soft'),
    ('gf_death',          'gf',          'death'),
    ('gf_filtered_soft',  'gf_filtered', 'soft'),
    ('gf_filtered_death', 'gf_filtered', 'death'),
]

v, w, c = generate_instance(N, M, ALPHA, seed=INSTANCE_SEED)
vl = v.tolist(); wl = w.tolist(); cl = c.tolist()
pj     = get_pj(w, c, PJ_SAMPLES, n=N, m=M, alpha=ALPHA,
                instance_seed=INSTANCE_SEED)
hill_p = toyoda_heuristic(vl, wl, cl, N, M)
print(f"Hill global p = {hill_p:.3f}\n")

summary = {}
for budget in BUDGET_VALUES:
    print(f"budget={budget}")
    result_file = {'label': f'Budget a={ALPHA} b={budget}',
                   'alpha': float(ALPHA), 'budget': int(budget),
                   'ref': REF_BEST}
    summary[budget] = {}
    for mkey, meth, pen in METHODS:
        runs = run_method(mkey, meth, pen, N_RUNS, budget,
                          N, M, vl, wl, cl, pj, hill_p, REF_BEST)
        result_file[mkey] = runs
        summary[budget][mkey] = np.mean([r['pct_of_best_final'] for r in runs])
    save_json(result_file,
              os.path.join(SAVE_DIR, f"results_Budget_a025_b{budget}_{timestamp}.json"))
    print()

print("\nSUMMARY")
header = f"{'Budget':>8}" + "".join(f"{k:>16}" for k,_,_ in METHODS)
print(header)
print("─"*len(header))
for b in BUDGET_VALUES:
    row = f"{b:>8}"
    for mkey,_,_ in METHODS:
        row += f"{summary[b][mkey]:>15.1f}%"
    print(row)

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/MKP_results'
TIMESTAMP = '20260507_181134'
BUDGETS   = [200, 500, 1000, 2000, 5000, 10000, 20000]

METHODS = {
    'uniform_soft':      {'label': 'Uniform + soft',          'color': '#888780', 'ls': '--',  'lw': 1.5, 'marker': 'o'},
    'hill_soft':         {'label': 'Hill (1999) + soft',      'color': '#BA7517', 'ls': '-.',  'lw': 1.5, 'marker': 's'},
    'gf_death':          {'label': 'GF-biased + death',       'color': '#378ADD', 'ls': '-',   'lw': 1.8, 'marker': '^'},
    'gf_filtered_soft':  {'label': 'Quality-filtered + soft', 'color': '#E74C3C', 'ls': '-',   'lw': 2.2, 'marker': 'D'},
    'gf_filtered_death': {'label': 'Quality-filtered + death','color': '#1D9E75', 'ls': '-.',  'lw': 1.8, 'marker': 'v'},
}

# Load all budget files
data = {}
for b in BUDGETS:
    fname = f"results_Budget_a025_b{b}_{TIMESTAMP}.json"
    with open(os.path.join(DRIVE_DIR, fname)) as f:
        d = json.load(f)
    for mkey in METHODS:
        if mkey not in data:
            data[mkey] = []
        runs   = d[mkey]
        finals = [r['pct_of_best_final'] for r in runs]
        data[mkey].append(np.mean(finals))

fig, ax = plt.subplots(figsize=(8, 5))

for mkey, style in METHODS.items():
    ax.plot(BUDGETS, data[mkey],
            label=style['label'],
            color=style['color'],
            ls=style['ls'],
            lw=style['lw'],
            marker=style['marker'],
            markersize=5)

ax.set_xscale('log')
ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS], fontsize=9)
ax.set_xlabel('Offspring budget (log scale)', fontsize=11)
ax.set_ylabel('Final fitness — % of reference best', fontsize=11)
ax.set_title(r'Solution quality vs evaluation budget'
             '\n' + r'$\alpha=0.25$, $n=100$, $m=3$, 20 runs averaged',
             fontsize=11)
ax.set_ylim(54, 100)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.2, which='both')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, f'fig_budget_sweep_{TIMESTAMP}.pdf'),
            bbox_inches='tight', dpi=300)
plt.savefig(os.path.join(DRIVE_DIR, f'fig_budget_sweep_{TIMESTAMP}.png'),
            bbox_inches='tight', dpi=300)
plt.show()
print("Saved.")

## Quality Filtering Seeding Sweep

In [ ]:
# ============================================================
# CANDIDATE POOL SWEEP
# Fixed: alpha=0.25, budget=2000 (where differences are sharpest)
# Varying: n_candidates in {50, 100, 200, 400, 800, 1600}
# Question: at what pool size does quality-filtering saturate?
# Needs: setup cell + cached p_j
# ============================================================

N, M, ALPHA   = 100, 3, 0.25
REF_BEST      = 2375
INSTANCE_SEED = 42
PJ_SAMPLES    = 20_000_000
N_RUNS        = 20
MAX_OFFSPRING = 2000
CANDIDATE_VALUES = [50, 100, 200, 400, 800, 1600]
timestamp     = datetime.now().strftime("%Y%m%d_%H%M%S")

v, w, c = generate_instance(N, M, ALPHA, seed=INSTANCE_SEED)
vl = v.tolist(); wl = w.tolist(); cl = c.tolist()
pj     = get_pj(w, c, PJ_SAMPLES, n=N, m=M, alpha=ALPHA,
                instance_seed=INSTANCE_SEED)
hill_p = toyoda_heuristic(vl, wl, cl, N, M)

# Run fixed baselines once
print("Fixed baselines (budget=2000):")
baseline_runs = {}
for mkey, meth, pen in [('uniform_soft','uniform','soft'),
                         ('gf_death','gf','death')]:
    baseline_runs[mkey] = run_method(
        mkey, meth, pen, N_RUNS, MAX_OFFSPRING,
        N, M, vl, wl, cl, pj, hill_p, REF_BEST)
print()

# Sweep candidate pool size
print(f"Candidate pool sweep (budget={MAX_OFFSPRING}):")
print(f"{'Candidates':>12} {'filt_soft':>12} {'filt_death':>12} "
      f"{'gen0_quality':>14} {'conv90_soft':>13}")
print("─"*68)

summary = {}
all_results = {
    'label':   'Candidate pool sweep',
    'alpha':   float(ALPHA),
    'budget':  MAX_OFFSPRING,
    'ref':     REF_BEST,
    'baselines': baseline_runs,
    'sweep':   []
}

for n_cand in CANDIDATE_VALUES:
    soft_runs  = run_method(
        f'filt_soft_c{n_cand}', 'gf_filtered', 'soft',
        N_RUNS, MAX_OFFSPRING,
        N, M, vl, wl, cl, pj, hill_p, REF_BEST,
        n_candidates=n_cand)
    death_runs = run_method(
        f'filt_death_c{n_cand}', 'gf_filtered', 'death',
        N_RUNS, MAX_OFFSPRING,
        N, M, vl, wl, cl, pj, hill_p, REF_BEST,
        n_candidates=n_cand)

    soft_finals  = [r['pct_of_best_final'] for r in soft_runs]
    death_finals = [r['pct_of_best_final'] for r in death_runs]
    gen0q        = [r['pct_of_best_gen0']  for r in soft_runs]
    c90s         = [r['convergence'].get('90') for r in soft_runs
                    if r['convergence'].get('90')]
    c90str       = f"{np.mean(c90s):.0f}" if c90s else "never"

    print(f"{n_cand:>12}  "
          f"{np.mean(soft_finals):>10.1f}%  "
          f"{np.mean(death_finals):>10.1f}%  "
          f"{np.mean(gen0q):>12.1f}%ref  "
          f"{c90str:>12}")

    all_results['sweep'].append({
        'n_candidates':      int(n_cand),
        'filt_soft_final':   float(np.mean(soft_finals)),
        'filt_death_final':  float(np.mean(death_finals)),
        'gen0_quality_mean': float(np.mean(gen0q)),
        'conv90_soft':       float(np.mean(c90s)) if c90s else None,
        'filt_soft_runs':    soft_runs,
        'filt_death_runs':   death_runs,
    })

save_json(all_results,
          os.path.join(SAVE_DIR,
                       f"results_CandidateSweep_a025_b{MAX_OFFSPRING}_{timestamp}.json"))

print()
print(f"Baselines at budget={MAX_OFFSPRING} for reference:")
for mkey in ['uniform_soft','gf_death']:
    runs   = baseline_runs[mkey]
    finals = [r['pct_of_best_final'] for r in runs]
    c90s   = [r['convergence'].get('90') for r in runs
              if r['convergence'].get('90')]
    print(f"  {mkey:>14}: {np.mean(finals):.1f}%  "
          f"conv90={f'{np.mean(c90s):.0f}' if c90s else 'never'}")

## Noisy Objective Experiment

In [ ]:
# ============================================================
# NOISY OBJECTIVE EXPERIMENT
# Robustness of GF-biased to objective uncertainty
# Consistent tracking: same fields as all other experiments
# No GPU needed — loads p_j from cache
# ============================================================

N, M, ALPHA   = 100, 3, 0.25
REF_BEST      = 2375
INSTANCE_SEED = 42
PJ_SAMPLES    = 20_000_000
N_RUNS        = 20
MAX_OFFSPRING = 1000
SIGMA_VALUES  = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
timestamp     = datetime.now().strftime("%Y%m%d_%H%M%S")

METHODS = [
    ('uniform_soft',     'uniform',      'soft'),
    ('hill_soft',        'hill',         'soft'),
    ('gf_death',         'gf',           'death'),
    ('gf_filtered_soft', 'gf_filtered',  'soft'),
]

v, w, c = generate_instance(N, M, ALPHA, seed=INSTANCE_SEED)
vl = v.tolist(); wl = w.tolist(); cl = c.tolist()
mean_v = float(np.mean(v))

pj     = get_pj(w, c, PJ_SAMPLES, n=N, m=M, alpha=ALPHA,
                instance_seed=INSTANCE_SEED)
hill_p_true = toyoda_heuristic(vl, wl, cl, N, M)
print(f"Hill global p (true) = {hill_p_true:.3f}")
print(f"Mean v = {mean_v:.1f}\n")

def add_noise(values, sigma_fraction, seed=0):
    rng   = np.random.RandomState(seed)
    noise = rng.normal(0, sigma_fraction * mean_v, len(values))
    return [max(0.1, values[j] + noise[j]) for j in range(len(values))]

all_results = {
    'label':        'Noisy objective experiment',
    'alpha':        float(ALPHA),
    'ref':          REF_BEST,
    'mean_v':       float(mean_v),
    'sigma_values': SIGMA_VALUES,
    'params': {
        'n': N, 'm': M,
        'instance_seed':  INSTANCE_SEED,
        'pj_seed':        42,
        'pj_samples':     PJ_SAMPLES,
        'pop_size':       50,
        'n_runs':         N_RUNS,
        'n_offspring':    MAX_OFFSPRING,
        'penalty_mult':   50.0,
        'convergence_thresholds': [50,60,70,80,90,95,98,99],
        'note': ('Noise corrupts v_j for Hill init and repair efficiency. '
                 'Fitness always evaluated on true v_j. '
                 'GF p_j computed from weights/capacities only.')
    },
    'sweep': []
}

print(f"{'Sigma':>7} {'uniform':>10} {'hill':>10} "
      f"{'gf_death':>10} {'gf_filt':>10} "
      f"{'hill_p':>8} {'conv90_gf':>11}")
print("─"*70)

for sigma in SIGMA_VALUES:
    noisy_vl     = add_noise(vl, sigma, seed=0)
    hill_p_noisy = toyoda_heuristic(noisy_vl, wl, cl, N, M)

    entry = {
        'sigma':          float(sigma),
        'hill_p_noisy':   float(hill_p_noisy),
        'hill_p_true':    float(hill_p_true),
        'methods':        {}
    }

    row_finals = {}
    row_c90    = {}

    for mkey, meth, pen in METHODS:
        hp = hill_p_noisy if meth == 'hill' else hill_p_true
        runs = []
        for r in range(N_RUNS):
            res = run_one_ga(
                N, M,
                vl,          # TRUE values for fitness always
                wl, cl,
                method=meth,
                penalty_type=pen,
                max_offspring=MAX_OFFSPRING,
                pj=pj,
                hill_p=hp,
                ref=REF_BEST,
                seed=r,
                track_diversity=False,
                track_every=50
            )
            runs.append(res)

        finals  = [r['pct_of_best_final']        for r in runs]
        gen0s   = [r['feasibility_rate_gen0']*100 for r in runs]
        c90s    = [r['convergence'].get('90')     for r in runs
                   if r['convergence'].get('90')]
        c90str  = f"{np.mean(c90s):.0f}" if c90s else "never"

        entry['methods'][mkey] = {
            'hill_p_used':    float(hp),
            'final_mean':     float(np.mean(finals)),
            'final_std':      float(np.std(finals)),
            'gen0_feas_mean': float(np.mean(gen0s)),
            'conv90_mean':    float(np.mean(c90s)) if c90s else None,
            'n_reached_90':   int(len(c90s)),
            'runs':           runs,
        }
        row_finals[mkey] = np.mean(finals)
        row_c90[mkey]    = c90str

    print(f"{sigma:>7.1f}  "
          f"{row_finals['uniform_soft']:>8.1f}%  "
          f"{row_finals['hill_soft']:>8.1f}%  "
          f"{row_finals['gf_death']:>8.1f}%  "
          f"{row_finals['gf_filtered_soft']:>8.1f}%  "
          f"{hill_p_noisy:>6.3f}  "
          f"{row_c90['gf_death']:>10}")

    all_results['sweep'].append(entry)

save_json(all_results,
          os.path.join(SAVE_DIR,
                       f"results_NoisyObjective_a025_{timestamp}.json"))

print(f"\nSaved. Fitness always on true v_j. "
      f"Noise only affects Hill init and repair ranking.")

## New Sampling Method


### Check against hand written examples

In [ ]:
# ============================================================
# IMPORTANCE-SAMPLING p_j ESTIMATOR
# Adds compute_pj_IS as an alternative to get_pj.
# Does NOT overwrite get_pj — both coexist.
# ============================================================
# Same mathematical definition (paper's rho^1_j, rho^0_j).
# Different sampling: q=alpha proposal + likelihood reweighting.
# Vectorized over items (no Python loop over j).
# Tracks Effective Sample Size (ESS) as diagnostic.
# ============================================================

# ============================================================
# SANITY CHECK — paper's worked example
# n=3, m=1, w=[3,2,4], W=5  →  p_j = [0.40, 0.40, 0.20]
# ============================================================
print("=" * 65)
print("SANITY CHECK — paper's n=3, m=1 worked example")
print("=" * 65)
W_s = np.array([[3, 2, 4]], dtype=np.float64)
C_s = np.array([5.0])
out_s = compute_pj_IS(W_s, C_s, n_samples=2_000_000, q=0.5,
                      batch_size=200_000, verbose=False)
print(f"  Expected p_j: [0.400, 0.400, 0.200]")
print(f"  Computed p_j: [{out_s['pj'][0]:.3f}, "
      f"{out_s['pj'][1]:.3f}, {out_s['pj'][2]:.3f}]")
err = np.max(np.abs(out_s['pj'].astype(np.float64)
                    - np.array([0.40, 0.40, 0.20])))
print(f"  Max abs error: {err:.4f}  "
      f"({'PASS' if err < 0.01 else 'FAIL'})")


# ============================================================
# TEST 1 — n=100, m=3, α=0.25  (compare new vs cached old p_j)
# ============================================================
print("\n" + "=" * 65)
print("TEST 1 — n=100, m=3, α=0.25  (compare old uniform vs new IS)")
print("=" * 65)

N1, M1, A1 = 100, 3, 0.25
v1, w1, c1 = generate_instance(N1, M1, A1, seed=42)

# Old pipeline (loads from cache if you have it)
pj_old = get_pj(w1, c1, 20_000_000, n=N1, m=M1, alpha=A1, instance_seed=42)

# New IS pipeline
out1   = compute_pj_IS(w1, c1, n_samples=10_000_000, q=None, batch_size=200_000)
pj_new = out1['pj']

fr_unif = feasibility_rate_check(np.full(N1, 0.5), w1, c1)
fr_old  = feasibility_rate_check(pj_old, w1, c1)
fr_new  = feasibility_rate_check(pj_new, w1, c1)

print(f"\n  Gen-0 feasibility rate (n_pop=10,000 samples):")
print(f"    Uniform p=0.5     : {100*fr_unif:6.2f}%")
print(f"    Old get_pj        : {100*fr_old:6.2f}%")
print(f"    New compute_pj_IS : {100*fr_new:6.2f}%")
print(f"\n  p_j vector comparison (should be similar at this scale):")
print(f"    Old:  min={pj_old.min():.3f}  mean={pj_old.mean():.3f}  "
      f"max={pj_old.max():.3f}  std={pj_old.std():.3f}")
print(f"    New:  min={pj_new.min():.3f}  mean={pj_new.mean():.3f}  "
      f"max={pj_new.max():.3f}  std={pj_new.std():.3f}")
print(f"    Per-item L1 diff   : {np.abs(pj_old - pj_new).mean():.4f}")
print(f"    Per-item Linf diff : {np.abs(pj_old - pj_new).max():.4f}")


# ============================================================
# TEST 2 — n=100, m=30, α=0.25  (TARGET BENCHMARK SCALE)
# Old get_pj would likely give 0/0 here. New IS is on its own.
# ============================================================
print("\n" + "=" * 65)
print("TEST 2 — n=100, m=30, α=0.25  (OR-Library benchmark scale)")
print("=" * 65)

N2, M2, A2 = 100, 30, 0.25
v2, w2, c2 = generate_instance(N2, M2, A2, seed=42)

out2 = compute_pj_IS(w2, c2, n_samples=10_000_000, q=None, batch_size=100_000)
pj2  = out2['pj']

fr_unif2 = feasibility_rate_check(np.full(N2, 0.5), w2, c2)
fr_new2  = feasibility_rate_check(pj2, w2, c2)

print(f"\n  Gen-0 feasibility rate (n_pop=10,000 samples):")
print(f"    Uniform p=0.5     : {100*fr_unif2:6.2f}%")
print(f"    New compute_pj_IS : {100*fr_new2:6.2f}%")
print(f"\n  KEY DIAGNOSTICS:")
print(f"    proposal feasibility : {100*out2['feas_rate_proposal']:6.3f}%  "
      f"(if 0%, q is too high)")
print(f"    ESS mean / batch     : {100*out2['ess_mean']/100_000:6.2f}%  "
      f"(if <1%, proposal mismatched)")
print(f"    p_j std (signal)     : {pj2.std():.4f}  "
      f"(if <0.01, signal washed out)")
print(f"    wall time            : {out2['time_sec']:6.1f}s")

In [ ]:
# ============================================================
# STABILITY CHECK — n=100, m=30, α=0.25
# Run compute_pj_IS 5 times with different torch seeds.
# If gen-0 feasibility stays in a tight band → robust.
# If p_j vectors agree across runs → estimates are real.
# If they vary wildly → ESS=10 is biting us.
# ============================================================

print("=" * 65)
print("STABILITY CHECK — n=100, m=30, α=0.25, 5 different seeds")
print("=" * 65)

N, M, A = 100, 30, 0.25
v, w, c = generate_instance(N, M, A, seed=42)

stability_runs = []
for run_seed in range(5):
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    out = compute_pj_IS(w, c, n_samples=10_000_000,
                        q=None, batch_size=100_000, verbose=False)
    pj = out['pj']
    fr = feasibility_rate_check(pj, w, c, n_pop=10_000, seed=0)
    stability_runs.append({
        'seed': run_seed,
        'pj':   pj,
        'fr':   fr,
        'ess':  out['ess_mean'],
        'feas_prop': out['feas_rate_proposal'],
    })
    print(f"  run {run_seed}: ESS={out['ess_mean']:>7.0f}  "
          f"prop_feas={100*out['feas_rate_proposal']:5.2f}%  "
          f"pj_std={pj.std():.4f}  "
          f"gen0_feas={100*fr:5.2f}%")

# ── Summary ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STABILITY SUMMARY")
print("=" * 65)

frs = np.array([r['fr'] for r in stability_runs]) * 100
print(f"  gen-0 feasibility across 5 runs:")
print(f"    mean  = {frs.mean():.2f}%")
print(f"    std   = {frs.std():.2f}%")
print(f"    range = [{frs.min():.2f}%, {frs.max():.2f}%]")

# ── Per-item p_j agreement across runs ─────────────────────
all_pj = np.stack([r['pj'] for r in stability_runs])  # (5, 100)
per_item_std = all_pj.std(axis=0)                     # (100,)
per_item_mean = all_pj.mean(axis=0)

print(f"\n  Per-item p_j std across 5 runs:")
print(f"    max  = {per_item_std.max():.4f}  "
      f"(item {int(per_item_std.argmax())})")
print(f"    mean = {per_item_std.mean():.4f}")
print(f"    median = {np.median(per_item_std):.4f}")

print(f"\n  Pairwise Linf distance between p_j vectors (max abs item-by-item):")
print("       " + "  ".join(f"  s{i}" for i in range(5)))
for i in range(5):
    print(f"   s{i} ", end="")
    for j in range(5):
        if j < i:
            print(f"  {'':>5}", end="")
        elif j == i:
            print(f"  {0.0:>5.3f}", end="")
        else:
            d = np.abs(stability_runs[i]['pj'] - stability_runs[j]['pj']).max()
            print(f"  {d:>5.3f}", end="")
    print()

print(f"\n  Interpretation:")
print(f"    - If gen-0 feas std < 2% across runs → robust outcome")
print(f"    - If per-item std mean < 0.02 → p_j is roughly stable")
print(f"    - If per-item std mean > 0.05 → ESS is hurting us; estimates noisy")

### Q sweep (q=alpha proposed, 0.30 optimal)

In [ ]:
# ============================================================
# Q SWEEP — n=100, m=30, α=0.25
# q below alpha = sparser samples, more feasible, but
#                 weights more concentrated → potentially worse ESS.
# Want to find sweet spot.
# ============================================================

print("=" * 65)
print("Q SWEEP — n=100, m=30, α=0.25, varying proposal probability")
print("=" * 65)

N, M, A = 100, 30, 0.25
v, w, c = generate_instance(N, M, A, seed=42)

q_values = [0.10, 0.15, 0.20, 0.25, 0.30]  # 0.25 is alpha
q_results = []

for q in q_values:
    torch.manual_seed(0)
    out = compute_pj_IS(w, c, n_samples=10_000_000,
                        q=q, batch_size=100_000, verbose=False)
    pj = out['pj']
    fr = feasibility_rate_check(pj, w, c, n_pop=10_000, seed=0)
    q_results.append({
        'q':         q,
        'pj':        pj,
        'fr':        fr,
        'ess':       out['ess_mean'],
        'feas_prop': out['feas_rate_proposal'],
        'time':      out['time_sec'],
    })
    print(f"  q={q:.2f}: prop_feas={100*out['feas_rate_proposal']:5.2f}%  "
          f"ESS={out['ess_mean']:>7.0f}  "
          f"pj_std={pj.std():.4f}  "
          f"gen0_feas={100*fr:5.2f}%  "
          f"time={out['time_sec']:.1f}s")

# ── Table ──────────────────────────────────────────────────
print("\n" + "=" * 65)
print("Q SWEEP SUMMARY TABLE")
print("=" * 65)
print(f"  {'q':>5}  {'prop_feas':>10}  {'ESS%':>7}  "
      f"{'pj_std':>7}  {'gen0_feas':>10}  {'time':>6}")
print(f"  {'-'*5}  {'-'*10}  {'-'*7}  {'-'*7}  {'-'*10}  {'-'*6}")
for r in q_results:
    print(f"  {r['q']:>5.2f}  "
          f"{100*r['feas_prop']:>9.2f}%  "
          f"{100*r['ess']/100_000:>6.2f}%  "
          f"{r['pj'].std():>7.4f}  "
          f"{100*r['fr']:>9.2f}%  "
          f"{r['time']:>5.1f}s")

# ── Best q by gen-0 feasibility ────────────────────────────
best = max(q_results, key=lambda r: r['fr'])
print(f"\n  Best q by gen-0 feasibility: q={best['q']:.2f}  "
      f"({100*best['fr']:.2f}%)")
best_ess = max(q_results, key=lambda r: r['ess'])
print(f"  Best q by ESS              : q={best_ess['q']:.2f}  "
      f"(ESS={best_ess['ess']:.0f})")

print(f"\n  Interpretation:")
print(f"    - Tradeoff: lower q = more feasible proposal, more weight concentration")
print(f"    - The interesting q is the one with highest ESS at decent gen-0 feas")

### IS vs Old method w/out IS

In [ ]:
# ============================================================
# OLD VS NEW — increasing m, α=0.25
# Where does get_pj's uniform sampling stop producing signal?
# Look for: (a) sharp drop in old gen-0 feas
#           (b) many items with old pj = 0 (no feasible samples seen)
# ============================================================

print("=" * 65)
print("OLD VS NEW — n=100, α=0.25, sweeping m")
print("=" * 65)
print("This will compute fresh p_j vectors. May take 5-10 min total.\n")

N, A = 100, 0.25
M_values = [3, 5, 10, 20, 30]
N_SAMPLES = 10_000_000

mvm_results = []
for M in M_values:
    print(f"\n{'-'*60}")
    print(f"  m = {M}")
    print(f"{'-'*60}")
    v, w, c = generate_instance(N, M, A, seed=42)

    # Old method
    torch.manual_seed(0)
    print(f"  [old get_pj]")
    pj_old = get_pj(w, c, N_SAMPLES, n=N, m=M, alpha=A,
                    instance_seed=42, force_recompute=True)
    fr_old   = feasibility_rate_check(pj_old, w, c, n_pop=10_000, seed=0)
    n_zeros  = int((pj_old < 1e-3).sum())
    n_uniform_fallback = int(np.abs(pj_old - 0.5).sum() < 1e-6)  # ~exactly 0.5
    n_in_band  = int(((pj_old >= 0.45) & (pj_old <= 0.55)).sum())  # near 0.5

    # New method
    torch.manual_seed(0)
    print(f"  [new compute_pj_IS]")
    out_new = compute_pj_IS(w, c, n_samples=N_SAMPLES,
                            q=None, batch_size=100_000, verbose=False)
    pj_new = out_new['pj']
    fr_new = feasibility_rate_check(pj_new, w, c, n_pop=10_000, seed=0)

    mvm_results.append({
        'm':        M,
        'pj_old':   pj_old,
        'fr_old':   fr_old,
        'n_zeros':  n_zeros,
        'n_band':   n_in_band,
        'pj_new':   pj_new,
        'fr_new':   fr_new,
        'ess_new':  out_new['ess_mean'],
        'feas_prop_new': out_new['feas_rate_proposal'],
    })

    print(f"\n  OLD: range=[{pj_old.min():.3f}, {pj_old.max():.3f}]  "
          f"std={pj_old.std():.4f}  "
          f"items_at_0={n_zeros}  items_near_0.5={n_in_band}  "
          f"gen0={100*fr_old:.2f}%")
    print(f"  NEW: range=[{pj_new.min():.3f}, {pj_new.max():.3f}]  "
          f"std={pj_new.std():.4f}  "
          f"ESS={out_new['ess_mean']:.0f}  prop_feas={100*out_new['feas_rate_proposal']:.2f}%  "
          f"gen0={100*fr_new:.2f}%")

# ── Summary table ──────────────────────────────────────────
print("\n" + "=" * 65)
print("OLD VS NEW SUMMARY")
print("=" * 65)
print(f"  {'m':>3} | {'OLD gen0':>9} | {'OLD zeros':>10} | {'OLD ~0.5':>9} | "
      f"{'OLD std':>8} || {'NEW gen0':>9} | {'NEW ESS':>8} | {'NEW std':>8}")
print(f"  {'-'*3}-+-{'-'*9}-+-{'-'*10}-+-{'-'*9}-+-{'-'*8}-++-"
      f"{'-'*9}-+-{'-'*8}-+-{'-'*8}")
for r in mvm_results:
    print(f"  {r['m']:>3} | "
          f"{100*r['fr_old']:>8.2f}% | "
          f"{r['n_zeros']:>9d}  | "
          f"{r['n_band']:>8d}  | "
          f"{r['pj_old'].std():>8.4f} || "
          f"{100*r['fr_new']:>8.2f}% | "
          f"{r['ess_new']:>7.0f}  | "
          f"{r['pj_new'].std():>8.4f}")

print(f"\n  What to look for:")
print(f"    - 'OLD zeros' jumping up = items the old method never saw work")
print(f"    - 'OLD ~0.5' jumping up  = items it had no signal on (fallback)")
print(f"    - 'OLD gen0' collapsing  = old method's wall")
print(f"    - 'NEW gen0' staying up across m = our method scales")

In [ ]:
# ============================================================
# VERIFICATION — what does "100% feasibility" actually mean?
# Reports avg items + avg value of feasible solutions.
# If OLD's "100% feas" is mostly empty knapsacks, we'll see it.
# ============================================================

def gen0_quality_check(p_j, values, weights, capacities,
                       n_pop=10_000, seed=0):
    p_j = np.asarray(p_j, dtype=np.float64)
    n = len(p_j)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    P   = torch.tensor(p_j,        dtype=torch.float64, device=device)
    W_t = torch.tensor(weights,    dtype=torch.float64, device=device)
    C_t = torch.tensor(capacities, dtype=torch.float64, device=device)
    V_t = torch.tensor(values,     dtype=torch.float64, device=device)
    g   = torch.Generator(device=device).manual_seed(seed)
    s = (torch.rand(n_pop, n, device=device, generator=g) < P).double()
    T     = s @ W_t.T
    feas  = (T <= C_t).all(dim=1)
    objs  = s @ V_t
    items = s.sum(dim=1)
    feas_idx = feas.nonzero().squeeze(-1)
    if len(feas_idx) == 0:
        return {'feas': 0.0, 'mean_val': 0.0, 'mean_items': 0.0}
    return {
        'feas':       feas.double().mean().item(),
        'mean_val':   objs[feas_idx].mean().item(),
        'mean_items': items[feas_idx].mean().item(),
    }

print("=" * 80)
print("VERIFICATION — feasibility AND solution quality across m")
print("=" * 80)

N, A = 100, 0.25
M_values = [3, 5, 10, 20, 30]

print(f"\n  {'m':>3} | {'METHOD':>6} | {'feas%':>6} | "
      f"{'avg items':>10} | {'avg value':>10}")
print(f"  {'-'*3}-+-{'-'*6}-+-{'-'*6}-+-{'-'*10}-+-{'-'*10}")

for M in M_values:
    v, w, c = generate_instance(N, M, A, seed=42)

    # Load OLD from cache (created by previous cell 3 run)
    fname = f"pj_n100_m{M}_a025_seed42_10M.json"
    with open(os.path.join(CACHE_DIR, fname)) as f:
        d = json.load(f)
    pj_old = np.array(d['pj'], dtype=np.float32)

    # Recompute NEW (fast on GPU)
    torch.manual_seed(0)
    out_new = compute_pj_IS(w, c, n_samples=10_000_000,
                            q=None, batch_size=100_000, verbose=False)
    pj_new = out_new['pj']

    qold = gen0_quality_check(pj_old, v, w, c)
    qnew = gen0_quality_check(pj_new, v, w, c)

    print(f"  {M:>3} | {'OLD':>6} | {100*qold['feas']:>5.1f}% | "
          f"{qold['mean_items']:>9.2f}  | {qold['mean_val']:>9.1f}")
    print(f"  {M:>3} | {'NEW':>6} | {100*qnew['feas']:>5.1f}% | "
          f"{qnew['mean_items']:>9.2f}  | {qnew['mean_val']:>9.1f}")
    print()

print("  Read this:")
print("    OLD avg items → 0 means it's producing empty knapsacks")
print("    OLD avg value → 0 means those 'feasible' solutions are worthless")
print("    NEW should show real items + real value at every m")


In [ ]:
# How many UNIQUE solutions in 10,000 samples? OLD m=5 should be ≈ 1.
N, M, A = 100, 5, 0.25
v, w, c = generate_instance(N, M, A, seed=42)
with open(os.path.join(CACHE_DIR, "pj_n100_m5_a025_seed42_10M.json")) as f:
    pj_old = np.array(json.load(f)['pj'], dtype=np.float32)
torch.manual_seed(0)
out = compute_pj_IS(w, c, n_samples=10_000_000, q=None,
                    batch_size=100_000, verbose=False)
pj_new = out['pj']

def count_unique(p_j, n_pop=10_000, seed=0):
    P = torch.tensor(p_j, dtype=torch.float64,
                     device='cuda' if torch.cuda.is_available() else 'cpu')
    g = torch.Generator(device=P.device).manual_seed(seed)
    s = (torch.rand(n_pop, len(p_j), device=P.device, generator=g)
         < P).int().cpu().numpy()
    return len({tuple(r) for r in s})

print(f"OLD m=5 unique solutions in 10,000 samples: {count_unique(pj_old):,}")
print(f"NEW m=5 unique solutions in 10,000 samples: {count_unique(pj_new):,}")

# OR Library Benchmark Tests

In [ ]:
# ============================================================
# FULL SWEEP WITH RESTART-ON-COLLAPSE — all 9 mknapcb files
# Self-contained. ~3 hours on T4. Saves to NEW subfolder so
# original orlib_tests/ data is preserved.
# ============================================================

import os, time, json, urllib.request
from datetime import datetime
from collections import Counter
import numpy as np
import torch

# ── Mount Drive (safe version) ───────────────────────────
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted.")

# ── Config ───────────────────────────────────────────────
SAVE_DIR        = '/content/drive/MyDrive/MKP_results'
SAVE_SUBFOLDER  = 'orlib_tests_restart'        # NEW folder — preserves originals
OR_LIB_URL      = 'http://people.brunel.ac.uk/~mastjjb/jeb/orlib/files/'
OR_LIB_DIR      = os.path.join(SAVE_DIR, 'or_library')
RESULTS_DIR     = os.path.join(SAVE_DIR, SAVE_SUBFOLDER)
os.makedirs(OR_LIB_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE          = 100_000
N_SAMPLES_MAIN      = 10_000_000
N_SAMPLES_PILOT     = 500_000
N_POP_CHECK         = 10_000
N_POP_PILOT         = 2_000
MAX_ATTEMPTS        = 5
COLLAPSE_THRESHOLD  = 0.01

RUN_ORDER = [
    'mknapcb1.txt',  # n=100  m=5
    'mknapcb4.txt',  # n=100  m=10
    'mknapcb7.txt',  # n=100  m=30
    'mknapcb2.txt',  # n=250  m=5
    'mknapcb5.txt',  # n=250  m=10
    'mknapcb8.txt',  # n=250  m=30
    'mknapcb3.txt',  # n=500  m=5
    'mknapcb6.txt',  # n=500  m=10
    'mknapcb9.txt',  # n=500  m=30
]
ORLIB_INDEX = {
    'mknapcb1.txt':(100,5),  'mknapcb2.txt':(250,5),  'mknapcb3.txt':(500,5),
    'mknapcb4.txt':(100,10), 'mknapcb5.txt':(250,10), 'mknapcb6.txt':(500,10),
    'mknapcb7.txt':(100,30), 'mknapcb8.txt':(250,30), 'mknapcb9.txt':(500,30),
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

# ── Helpers ──────────────────────────────────────────────
def download_orlib(filename):
    local = os.path.join(OR_LIB_DIR, filename)
    if not os.path.exists(local):
        urllib.request.urlretrieve(OR_LIB_URL + filename, local)
    return local

def parse_mknapcb(filepath):
    with open(filepath) as f:
        nums = [float(x) for x in f.read().split()]
    idx = 0
    n_inst = int(nums[idx]); idx += 1
    out = []
    for iid in range(n_inst):
        n   = int(nums[idx]);   idx += 1
        m   = int(nums[idx]);   idx += 1
        opt = float(nums[idx]); idx += 1
        v   = np.array(nums[idx:idx+n], dtype=np.float64); idx += n
        w   = np.zeros((m, n), dtype=np.float64)
        for i in range(m):
            w[i] = nums[idx:idx+n]; idx += n
        c   = np.array(nums[idx:idx+m], dtype=np.float64); idx += m
        out.append({'id': iid, 'n': n, 'm': m, 'opt': opt,
                    'v': v, 'w': w, 'c': c})
    return out

def hill_init(values, weights, capacities, n, m):
    remaining = list(range(n))
    cap = list(capacities)
    selected = []
    while remaining:
        best_j, best_eff = None, -1.0
        for j in remaining:
            if not all(weights[i][j] <= cap[i] for i in range(m)):
                continue
            denom = sum(weights[i][j] / max(cap[i], 1e-9) for i in range(m))
            eff = values[j] / (denom + 1e-9)
            if eff > best_eff:
                best_eff, best_j = eff, j
        if best_j is None: break
        selected.append(best_j)
        for i in range(m): cap[i] -= weights[i][best_j]
        remaining.remove(best_j)
    return len(selected) / n

def compute_pj_IS(weights, capacities, n_samples, q):
    W = np.asarray(weights, dtype=np.float64)
    C = np.asarray(capacities, dtype=np.float64)
    m, n = W.shape
    q = float(np.clip(q, 1e-3, 1 - 1e-3))
    Wt = torch.tensor(W, dtype=torch.float64, device=DEVICE)
    Ct = torch.tensor(C, dtype=torch.float64, device=DEVICE)
    lq, l1q = float(np.log(q)), float(np.log(1.0 - q))
    sw_inc = torch.zeros(n, dtype=torch.float64, device=DEVICE)
    sw_exc = torch.zeros(n, dtype=torch.float64, device=DEVICE)
    fp     = torch.zeros((), dtype=torch.float64, device=DEVICE)
    ess_l, done = [], 0
    t0 = time.time()
    for _ in range((n_samples + BATCH_SIZE - 1) // BATCH_SIZE):
        bs = min(BATCH_SIZE, n_samples - done)
        s  = (torch.rand(bs, n, device=DEVICE) < q).double()
        k  = s.sum(dim=1)
        lw = -(k * lq + (n - k) * l1q); lw = lw - lw.max()
        ww = torch.exp(lw)
        T  = s @ Wt.T
        ci = torch.ones(bs, n, dtype=torch.bool, device=DEVICE)
        ce = torch.ones(bs, n, dtype=torch.bool, device=DEVICE)
        ff = torch.ones(bs,    dtype=torch.bool, device=DEVICE)
        for i in range(m):
            Ti = T[:, i:i+1]; Wi = Wt[i:i+1, :]; Ci = float(Ct[i].item())
            ci &= (Ti <= Ci - Wi * (1 - s))
            ce &= (Ti <= Ci + Wi * s)
            ff &= (T[:, i] <= Ci)
        we = ww.unsqueeze(1)
        sw_inc += (we * ci.double()).sum(0)
        sw_exc += (we * ce.double()).sum(0)
        fp     += ff.double().sum()
        ess_l.append((ww.sum()**2 / (ww**2).sum()).item())
        done += bs
    si = sw_inc.cpu().numpy(); se = sw_exc.cpu().numpy()
    eps = 1e-12
    pj = (si + eps) / (si + se + 2*eps)
    return {'pj': pj.astype(np.float32),
            'ess_mean': float(np.mean(ess_l)),
            'feas_prop': float(fp.cpu().item()) / n_samples,
            'time_sec': time.time() - t0}

def gen0_check(p_or_scalar, values, weights, capacities, n_pop=N_POP_CHECK, seed=0):
    p = np.asarray(p_or_scalar, dtype=np.float64)
    if p.ndim == 0:
        p = np.full(len(values), float(p))
    n = len(p)
    P  = torch.tensor(p,          dtype=torch.float64, device=DEVICE)
    Wt = torch.tensor(weights,    dtype=torch.float64, device=DEVICE)
    Ct = torch.tensor(capacities, dtype=torch.float64, device=DEVICE)
    Vt = torch.tensor(values,     dtype=torch.float64, device=DEVICE)
    g  = torch.Generator(device=DEVICE).manual_seed(seed)
    s  = (torch.rand(n_pop, n, device=DEVICE, generator=g) < P).double()
    T  = s @ Wt.T
    feas = (T <= Ct).all(dim=1)
    rate = feas.double().mean().item()
    idx = feas.nonzero().squeeze(-1)
    if len(idx) == 0:
        return {'feas': rate, 'n_feas': 0, 'items': 0.0, 'items_std': 0.0,
                'val_mean': 0.0, 'val_max': 0.0, 'val_min': 0.0,
                'val_std': 0.0, 'ham': 0.0, 'unique': 0}
    sf = s[idx]
    objs = (sf @ Vt).cpu().numpy()
    itms = sf.sum(dim=1).cpu().numpy()
    freq = sf.mean(dim=0)
    ham = (2.0 * (freq * (1.0 - freq)).sum()).item()
    uniq = len({row.tobytes() for row in sf.cpu().numpy().astype(np.int8)})
    return {'feas': rate, 'n_feas': int(len(idx)),
            'items': float(itms.mean()), 'items_std': float(itms.std()),
            'val_mean': float(objs.mean()), 'val_max': float(objs.max()),
            'val_min': float(objs.min()),  'val_std': float(objs.std()),
            'ham': ham, 'unique': uniq}

def get_q_candidates(alpha):
    return sorted({
        round(max(0.05, alpha - 0.10), 3),
        round(max(0.05, alpha - 0.05), 3),
        round(alpha, 3),
        round(min(0.95, alpha + 0.05), 3),
    })

def compute_pj_with_restart(weights, capacities, values, alpha,
                             pilot_size=N_SAMPLES_PILOT,
                             main_size=N_SAMPLES_MAIN,
                             max_attempts=MAX_ATTEMPTS,
                             collapse_threshold=COLLAPSE_THRESHOLD):
    t0 = time.time()
    history = []
    best = None
    for attempt in range(max_attempts):
        torch.manual_seed(attempt)
        np.random.seed(attempt)
        # Pilot
        cands = get_q_candidates(alpha)
        pilot = []
        for q in cands:
            out_p = compute_pj_IS(weights, capacities, n_samples=pilot_size, q=q)
            chk_p = gen0_check(out_p['pj'], values, weights, capacities,
                                n_pop=N_POP_PILOT, seed=attempt)
            pilot.append({'q': q, 'feas': chk_p['feas']})
        best_p = max(pilot, key=lambda x: x['feas'])
        q_star, pilot_feas = best_p['q'], best_p['feas']
        # Pilot collapse → restart
        if pilot_feas < collapse_threshold:
            history.append({'attempt': attempt, 'q_star': q_star,
                            'pilot_feas': pilot_feas, 'main_feas': None,
                            'stopped_at': 'pilot'})
            continue
        # Main run
        out = compute_pj_IS(weights, capacities, n_samples=main_size, q=q_star)
        chk = gen0_check(out['pj'], values, weights, capacities,
                         n_pop=N_POP_CHECK, seed=attempt)
        rec = {'attempt': attempt, 'q_star': q_star,
               'pilot_feas': pilot_feas, 'main_feas': chk['feas'],
               'pj': out['pj'], 'ess': out['ess_mean'],
               'feas_prop': out['feas_prop'], 'check': chk,
               'stopped_at': 'main'}
        history.append(rec)
        if best is None or chk['feas'] > best['check']['feas']:
            best = rec
        # Success → stop
        if chk['feas'] >= collapse_threshold:
            return {'pj': out['pj'], 'q_star': q_star,
                    'attempt': attempt + 1, 'feas': chk['feas'],
                    'check': chk, 'ess': out['ess_mean'],
                    'feas_prop': out['feas_prop'],
                    'history': history, 'time_sec': time.time() - t0}
    # All exhausted — return best (or fallback if all pilots collapsed)
    if best is None:
        pj_fb = np.full(len(values), alpha, dtype=np.float32)
        return {'pj': pj_fb, 'q_star': alpha, 'attempt': max_attempts,
                'feas': 0.0, 'check': None, 'ess': 0.0, 'feas_prop': 0.0,
                'history': history, 'time_sec': time.time() - t0}
    return {'pj': best['pj'], 'q_star': best['q_star'],
            'attempt': max_attempts, 'feas': best['check']['feas'],
            'check': best['check'], 'ess': best['ess'],
            'feas_prop': best['feas_prop'],
            'history': history, 'time_sec': time.time() - t0}

def pct(a, b):
    if abs(b) < 1e-9: return float('inf') if a > 1e-9 else 0.0
    return (a - b) / b * 100.0

def fmt(x):
    if x == float('inf'):  return '   +inf%'
    if x == float('-inf'): return '   -inf%'
    return f"{x:+7.1f}%"

# ════════════════════════════════════════════════════════
# OUTER LOOP
# ════════════════════════════════════════════════════════
overall_start = time.time()
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Files  : {len(RUN_ORDER)}")
print(f"Method : adaptive q pilot + restart-on-collapse (max {MAX_ATTEMPTS} attempts)")
print(f"Output : {RESULTS_DIR}\n")

master_log = []

for file_idx, TARGET_FILE in enumerate(RUN_ORDER):
    file_start = time.time()
    expected_n, expected_m = ORLIB_INDEX[TARGET_FILE]

    print("█" * 175)
    print(f"█ FILE {file_idx+1}/{len(RUN_ORDER)}: {TARGET_FILE} "
          f"(n={expected_n}, m={expected_m}) [{datetime.now().strftime('%H:%M:%S')}]")
    print("█" * 175)

    try:
        filepath  = download_orlib(TARGET_FILE)
        instances = parse_mknapcb(filepath)
        n_actual, m_actual = instances[0]['n'], instances[0]['m']
        print(f"Loaded: {len(instances)} instances")

        print()
        print("=" * 175)
        print(f" GEN-0 + QUALITY + DIVERSITY — {TARGET_FILE}  (restart-on-collapse)")
        print(f" Per method:  feas%/items/val_mean/val_max/hamming/unique")
        print("=" * 175)
        hdr = (f" {'#':>3} {'α':>5} {'q*':>5} {'tries':>5} || "
               f"{'UN feas/itm/vmean/vmax/ham/uniq':>40} || "
               f"{'HI feas/itm/vmean/vmax/ham/uniq':>40} || "
               f"{'GF feas/itm/vmean/vmax/ham/uniq':>40} || "
               f"{'h_p':>5} {'ESS':>7} {'prop':>5}")
        print(hdr)
        print(" " + "-" * 173)

        file_results = []
        for inst in instances:
            n, m = inst['n'], inst['m']
            v, w, c = inst['v'], inst['w'], inst['c']
            alpha = float((c / w.sum(axis=1)).mean())

            # GF with restart
            res = compute_pj_with_restart(w, c, v, alpha)
            pj  = res['pj']
            chk_gf   = res['check']
            ess_gf   = res['ess']
            attempts = res['attempt']
            q_star   = res['q_star']
            prop     = res['feas_prop']

            # Hill + Uniform baselines
            hill_p   = hill_init(v.tolist(), w.tolist(), c.tolist(), n, m)
            chk_hill = gen0_check(hill_p, v, w, c, n_pop=N_POP_CHECK, seed=0)
            chk_unif = gen0_check(0.5,    v, w, c, n_pop=N_POP_CHECK, seed=0)

            def render(q):
                if q is None:
                    return f"{'0.0%/0.0/0/0/0.0/0':>40}"
                return (f"{100*q['feas']:>5.1f}%/{q['items']:>4.1f}/"
                        f"{q['val_mean']:>6.0f}/{q['val_max']:>6.0f}/"
                        f"{q['ham']:>4.1f}/{q['unique']:>5d}")

            print(f" {inst['id']:>3} {alpha:>5.3f} {q_star:>5.3f} "
                  f"{attempts:>5} || "
                  f"{render(chk_unif):>40} || "
                  f"{render(chk_hill):>40} || "
                  f"{render(chk_gf):>40} || "
                  f"{hill_p:>5.3f} {ess_gf:>7.0f} {100*prop:>4.1f}%")

            file_results.append({
                'instance_id': inst['id'], 'n': n, 'm': m,
                'alpha': alpha, 'optimum': inst['opt'],
                'unif': chk_unif, 'hill': chk_hill, 'gf': chk_gf,
                'hill_p': hill_p,
                'pj_std': float(pj.std()),
                'ess_mean': ess_gf,
                'q_chosen': q_star,
                'feas_rate_proposal': prop,
                'attempts': attempts,
                'time_sec': res['time_sec'],
                'restart_history': [{k: v for k, v in h.items()
                                     if k not in ('pj', 'check')}
                                    for h in res['history']],
                'pj': pj.tolist(),
            })

        # Per-file summary
        print()
        print("=" * 110)
        print(f" SUMMARY BY ALPHA — {TARGET_FILE}")
        print("=" * 110)
        METRICS = ['feas','items','val_mean','val_max','val_std','ham','unique']
        for lo, hi, label in [(0.20,0.35,'~0.25'),(0.40,0.60,'~0.50'),(0.65,0.80,'~0.75')]:
            bucket = [r for r in file_results if lo <= r['alpha'] <= hi]
            if not bucket: continue
            avg = lambda mthd, key: np.mean([r[mthd][key] for r in bucket])
            print(f" alpha {label}  ({len(bucket)} instances):")
            for mthd in ['unif','hill','gf']:
                print(f"  {mthd:>5} | "
                      f"feas={100*avg(mthd,'feas'):>5.1f}% "
                      f"items={avg(mthd,'items'):>5.1f} "
                      f"vmean={avg(mthd,'val_mean'):>7.0f} "
                      f"vmax={avg(mthd,'val_max'):>7.0f} "
                      f"ham={avg(mthd,'ham'):>5.1f} "
                      f"uniq={avg(mthd,'unique'):>5.0f}")
            print(f"   GF vs Hill: feas {fmt(pct(avg('gf','feas'),avg('hill','feas')))}, "
                  f"val_max {fmt(pct(avg('gf','val_max'),avg('hill','val_max')))}, "
                  f"items {fmt(pct(avg('gf','items'),avg('hill','items')))}")
            atts = [r['attempts'] for r in bucket]
            qs   = [r['q_chosen'] for r in bucket]
            wins = sum(1 for r in bucket if r['gf']['feas'] > r['hill']['feas'])
            print(f"   GF wins on {wins}/{len(bucket)} | "
                  f"avg attempts: {np.mean(atts):.2f} (max {max(atts)}) | "
                  f"q* mode: {Counter(qs).most_common(1)[0][0]:.3f}")
            print()

        # Save per file
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output = {
            'label':         f'OR-Library {TARGET_FILE} (adaptive q + restart-on-collapse)',
            'file':          TARGET_FILE,
            'n':             n_actual, 'm': m_actual,
            'n_samples':     N_SAMPLES_MAIN,
            'pilot_size':    N_SAMPLES_PILOT,
            'max_attempts':  MAX_ATTEMPTS,
            'collapse_threshold': COLLAPSE_THRESHOLD,
            'n_pop_check':   N_POP_CHECK,
            'timestamp':     timestamp,
            'wall_time_sec': time.time() - file_start,
            'results':       file_results,
        }
        out_path = os.path.join(
            RESULTS_DIR,
            f"results_{TARGET_FILE.replace('.txt','')}_restart_{timestamp}.json"
        )
        with open(out_path, 'w') as fp:
            json.dump(output, fp, default=lambda x: x.tolist()
                      if isinstance(x, np.ndarray) else float(x))
        print(f" Saved   : {out_path}")
        print(f" File time: {(time.time() - file_start)/60:.1f} min "
              f"({(time.time() - file_start)/len(file_results):.1f}s per instance)")

        master_log.append({
            'file': TARGET_FILE, 'n': n_actual, 'm': m_actual,
            'n_instances': len(file_results),
            'wall_time_sec': time.time() - file_start,
            'mean_attempts': np.mean([r['attempts'] for r in file_results]),
            'saved_to': out_path,
        })

        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"  ERROR on {TARGET_FILE}: {e}")
        import traceback; traceback.print_exc()
        master_log.append({'file': TARGET_FILE, 'error': str(e)})

# Master summary
total_min = (time.time() - overall_start) / 60
print()
print("█" * 95)
print(f"█  COMPLETED — total wall time {total_min:.1f} min  "
      f"({datetime.now().strftime('%Y-%m-%d %H:%M:%S')})")
print("█" * 95)
print()
print(f"  {'File':>16}  {'(n, m)':>10}  {'instances':>9}  "
      f"{'avg attempts':>12}  {'time (min)':>11}  status")
print(f"  {'-'*16}  {'-'*10}  {'-'*9}  {'-'*12}  {'-'*11}  {'-'*8}")
for log in master_log:
    if 'error' in log:
        print(f"  {log['file']:>16}  {'-':>10}  {'-':>9}  {'-':>12}  "
              f"{'-':>11}  ERROR: {log['error']}")
    else:
        print(f"  {log['file']:>16}  ({log['n']}, {log['m']:>2})  "
              f"{log['n_instances']:>9}  {log['mean_attempts']:>12.2f}  "
              f"{log['wall_time_sec']/60:>11.1f}  ok")
print()
print(f"  Results saved to: {RESULTS_DIR}")
print(f"  Original (pre-restart) data preserved in: {os.path.join(SAVE_DIR, 'orlib_tests')}")